# 05 — Tubularity Analysis

Computes **S_tight** (tightness) and **S_cross** (crossing) metrics across Retina, V1, and FNN layers with 30 bootstrap resamples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
sys.path.insert(0, '..')
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from src.tubes_utils import preprocess_curves, run_clustering, compute_cluster_metrics, aggregate_metrics, plot_scenario_overview_proper


## Config

In [ ]:
use_ground_truth_labels = True

data_paths = [
    os.path.abspath('../data/sampled/tensor4d_retina.npy'),
    os.path.abspath('../data/sampled/tensor4d_V1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_inputs0_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1.npy'),
    #os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_recurrentout_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_seed2.npy'),
]

exp_names = ["Retina", "V1", "Enc1", "Enc13", "Rec", "RecOut", "Readout", "Output"]

# ── Bootstrap / tubularity parameters ───────────────────────────────────────
N_BOOTSTRAP = 30     # outer bootstrap resamples (neuron subsampling)
M = 150              # number of arclength resampling points per curve
B = 100              # inner bootstrap resamples inside compute_cluster_metrics
q = 0.9              # quantile for S_tight tube radius
smoothing = 2.0      # centerline spline smoothing factor


## Compute bootstrap tubularity scores

In [ ]:
result_dict = {}
bio_t, art_t, bio_c, art_c, enc_c, later_c = [], [], [], [], [], []

for i, data_path in enumerate(data_paths):
    data = np.load(data_path)

    if "position" in data_path:
        data = data[:, :6]
    elif "fnn" in data_path:
        data = data[:, np.array([0, 6, 7, 8, 9, 10])]
    print(f"Loaded {exp_names[i]}: {data.shape}")

    n_neurons, stim, dirs, time = data.shape
    orig_data = np.transpose(data, (1, 2, 3, 0)).reshape(stim * dirs * time, -1)

    bootstrapped_t, bootstrapped_c = [], []
    for j in range(30):
        rng = np.random.default_rng()
        data_b = rng.choice(orig_data, axis=1, size=orig_data.shape[1], replace=True)

        pipeline = Pipeline([('scaling', StandardScaler()), ('pca', PCA(n_components=10))])
        pca_traj_result = pipeline.fit_transform(data_b)
        data_b = pca_traj_result.reshape(stim * dirs, time, -1)

        curves_raw = list(data_b)
        M = 150
        curves = preprocess_curves(curves_raw, M=M)
        labels, _ = run_clustering(curves, metric="H1", alpha=0.5, min_cluster_size=4)

        if use_ground_truth_labels:
            labels = np.repeat(np.arange(stim), dirs)

        results = compute_cluster_metrics(curves, labels, smoothing=2.0, q=0.9, B=100)
        t_avg, c_avg, t_var, c_var = aggregate_metrics(results)
        bootstrapped_t.append(t_avg)
        bootstrapped_c.append(c_avg)

        if i < 2:
            bio_t.append(t_avg); bio_c.append(c_avg)
        else:
            art_t.append(t_avg); art_c.append(c_avg)
        if 2 <= i < 4:
            enc_c.append(c_avg)
        elif i >= 4:
            later_c.append(c_avg)

        print(f"  rep {j:2d}: S_tight={t_avg:.3f}  S_cross={c_avg:.3f}")

    if results:
        plot_scenario_overview_proper(curves_raw, curves, results, labels,
                                     title="curves + fitted centerlines", layer=exp_names[i])
        result_dict[exp_names[i]] = {
            "s_tight":     np.mean(bootstrapped_t),
            "s_tight_var": np.var(bootstrapped_t),
            "s_cross":     np.mean(bootstrapped_c),
            "s_cross_var": np.var(bootstrapped_c),
            "clusters":    len(np.unique(labels)),
        }


## Statistical tests

In [ ]:
from scipy import stats

u1, p1 = stats.mannwhitneyu(bio_t, art_t, alternative='less')
u2, p2 = stats.mannwhitneyu(bio_c, art_c, alternative='less')
u3, p3 = stats.mannwhitneyu(later_c, enc_c, alternative='less')

print(f"S_tight  bio<art:   U={u1:.1f}  p={p1:.4f}")
print(f"S_cross  bio<art:   U={u2:.1f}  p={p2:.4f}")
print(f"S_cross  later<enc: U={u3:.1f}  p={p3:.4f}")


## Plot

In [ ]:
from tueplots import bundles, axes as tpaxes
from tueplots.constants.color import rgb

exp_names_list = list(result_dict.keys())
s_tight     = [result_dict[n]["s_tight"]     for n in exp_names_list]
s_tight_var = [result_dict[n]["s_tight_var"] for n in exp_names_list]
s_cross     = [result_dict[n]["s_cross"]     for n in exp_names_list]
s_cross_var = [result_dict[n]["s_cross_var"] for n in exp_names_list]

x = np.arange(len(exp_names_list))
width = 0.35

with plt.rc_context({**bundles.neurips2024(), **tpaxes.lines()}):
    fig, ax1 = plt.subplots(figsize=(10, 6))

    ax1.bar(x - width/2, s_tight, width,
            yerr=np.sqrt(s_tight_var) / np.sqrt(100),
            label='Tightness', color='red', capsize=5)
    ax1.set_ylabel('Tightness', color='red', fontsize=20)
    ax1.tick_params(axis='y', labelcolor='red', labelsize=20)

    ax2 = ax1.twinx()
    ax2.bar(x + width/2, s_cross, width,
            yerr=np.sqrt(s_cross_var) / np.sqrt(100),
            label='Crossing', color='blue', capsize=5)
    ax2.set_ylabel('Crossing', color='blue', fontsize=20)
    ax2.tick_params(axis='y', labelcolor='blue', labelsize=20)

    ax1.set_xticks(x)
    ax1.set_xticklabels(exp_names_list, rotation=45, ha='right', fontsize=20)
    ax1.set_xlim(-0.5, len(exp_names_list) - 0.5)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=20)

    ax1.axvspan(-0.5, 1.5, color='green', alpha=0.1)
    ax1.text(0.5, ax1.get_ylim()[1] * 0.95, 'Biological', ha='center', va='top', fontsize=16, color='green')
    ax1.axvspan(1.5, 7.5, color='purple', alpha=0.1)
    ax1.text(3.5, ax1.get_ylim()[1] * 0.95, 'FNN', ha='center', va='top', fontsize=16, color='purple')

    plt.tight_layout()
    os.makedirs('../fig/tubularity', exist_ok=True)
    suffix = 'ground_truth' if use_ground_truth_labels else 'hdbscan'
    plt.savefig(f'../fig/tubularity/metrics_{suffix}.pdf')
    plt.show()
